# Phase 07: Universal Balanced ViT Training & Cross-Generative Evaluation (EXP-04)

## Subtitle + Purpose
Full end-to-end interactive training and cross-paradigm evaluation pipeline for **DINOv3 ViT** on the **Universal Perfectly-Balanced Dataset Splits V4** (`train_v4_universal_balanced.csv`, `val_v4_universal_balanced.csv`, `test_v4_universal_balanced.csv`, `test_balanced.csv`). This notebook implements **Layer-wise Learning Rate Decay (LLRD)**, **Label Smoothing Cross Entropy**, **Model Exponential Moving Average (EMA)**, **Test-Time Augmentation (TTA)**, and detailed per-method performance breakdowns across all 38 generative algorithms.

## Roadmap Table
| Step | Description | What it does | Import path |
|:---|:---|:---|:---|
| 1 | Environment Setup & Hyperparameters | Sets seeds, CUDA device, and hyperparameters | `torch`, `numpy`, `pathlib` |
| 2 | Dataset Loaders & Augmentations | Builds high-throughput PyTorch DataLoaders for V4 splits | `torch.utils.data`, `torchvision.transforms` |
| 3 | Pretrained DINOv3 ViT & Head Init | Loads DINOv3 ViT-B/14 backbone and enhanced 384-dim classifier head | `src.models.dinov3_vit` |
| 4 | LLRD & Cosine Scheduler Setup | Configures layer-wise LR decay ($0.8^L$) and AdamW optimizer | `src.training.losses`, `torch.optim` |
| 5 | Evaluation Engine & Calibration | Calibrates multi-metric evaluation engine with optimal threshold tau* | `sklearn.metrics` |
| 6 | Interactive Training Loop with EMA | Executes 10-epoch training with mixed precision (AMP) and EMA saving | `src.training.ema`, `tqdm` |
| 7 | Convergence Curves Visualization | Plots Loss, Accuracy, and ROC-AUC convergence across epochs | `matplotlib.pyplot` |
| 8 | Validation V4 Multi-Metric Evaluation | Computes optimal threshold tau*, ROC curve, and confusion matrix | `sklearn.metrics` |
| 9 | Test V4 Evaluation with TTA | Evaluates EMA model on independent Test V4 set (5,000 samples) | `src.eval.tta` |
| 10 | Frozen Test Balanced Benchmark Eval | Evaluates on the frozen 4,134-image international benchmark suite | `sklearn.metrics` |
| 11 | Granular 38-Method Breakdown & Report | Computes per-method accuracy across all generative paradigms | `pandas`, `matplotlib.pyplot` |
---

## References
- Rules: `[NOTEBOOK_HEADER_CONVENTION.md](../agents/rules/NOTEBOOK_HEADER_CONVENTION.md)`, `[LOGGING_CHECKPOINT_RULES.md](../agents/rules/LOGGING_CHECKPOINT_RULES.md)`
- Training Script: `[src/training/train_exp04.py](../src/training/train_exp04.py)`
- Splits: `[data/splits/train_v4_universal_balanced.csv](../data/splits/train_v4_universal_balanced.csv)`
- Checkpoints: `[experiments/checkpoints/exp04_dinov3_v4universal/](../experiments/checkpoints/exp04_dinov3_v4universal/)`


In [4]:
# Step 1: Environment Setup & Hyperparameters
import os
import sys
import time
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)

PROJECT_ROOT = Path("..").resolve() if Path("..").resolve().name == "deepfake-ViT" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
PLOTS_DIR = PROJECT_ROOT / "experiments" / "plots"
RESULTS_DIR = PROJECT_ROOT / "experiments" / "results"
CKPT_DIR = PROJECT_ROOT / "experiments" / "checkpoints" / "exp04_dinov3_v4universal"

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"📁 Project Root: {PROJECT_ROOT}")
print(f"⚡ Compute Device: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"💾 Checkpoints Output: {CKPT_DIR}")

# Hyperparameters
CONFIG = {
    "epochs": 10,
    "batch_size": 16,
    "base_lr": 1e-5,
    "head_lr": 1e-3,
    "decay_rate": 0.8,
    "weight_decay": 0.05,
    "label_smoothing": 0.05,
    "ema_decay": 0.999,
    "num_workers": 4,
    "img_size": 256,
    "hidden_dim": 384,
    "dropout": 0.2
}
print(f"⚙️ Config: {json.dumps(CONFIG, indent=2)}")


📁 Project Root: /workspace/hoangtuan/deepfake-ViT
⚡ Compute Device: cuda (NVIDIA GeForce RTX 3060)
💾 Checkpoints Output: /workspace/hoangtuan/deepfake-ViT/experiments/checkpoints/exp04_dinov3_v4universal
⚙️ Config: {
  "epochs": 10,
  "batch_size": 16,
  "base_lr": 1e-05,
  "head_lr": 0.001,
  "decay_rate": 0.8,
  "weight_decay": 0.05,
  "label_smoothing": 0.05,
  "ema_decay": 0.999,
  "num_workers": 4,
  "img_size": 256,
  "hidden_dim": 384,
  "dropout": 0.2
}


In [5]:
# Step 2: Dataset Loaders & Advanced Augmentations
from src.training.losses import LabelSmoothingCrossEntropy
from src.training.ema import ModelEMA
from src.eval.tta import predict_batch_with_tta

train_transforms = T.Compose([
    T.Resize((CONFIG["img_size"], CONFIG["img_size"]), interpolation=T.InterpolationMode.BICUBIC),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomApply([T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05)], p=0.5),
    T.RandomApply([T.GaussianBlur(kernel_size=(3, 5), sigma=(0.1, 2.0))], p=0.3),
    T.RandomApply([T.RandomAdjustSharpness(sharpness_factor=2.0)], p=0.3),
    T.RandomApply([T.RandomAutocontrast()], p=0.2),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transforms = T.Compose([
    T.Resize((CONFIG["img_size"], CONFIG["img_size"]), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class UniversalDeepfakeDataset(Dataset):
    def __init__(self, csv_path, transform=None, max_samples=None):
        df = pd.read_csv(csv_path)
        if max_samples and len(df) > max_samples:
            df = df.sample(n=max_samples, random_state=SEED).reset_index(drop=True)
        self.paths = df["path"].values
        self.labels = df["label"].values
        self.methods = df["method"].values if "method" in df.columns else np.array(["unknown"] * len(df))
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        label = self.labels[idx]
        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            img = Image.new("RGB", (CONFIG["img_size"], CONFIG["img_size"]), (0, 0, 0))
        tensor = self.transform(img) if self.transform else T.ToTensor()(img)
        return tensor, torch.tensor(label, dtype=torch.long), idx

# Instantiate datasets
train_csv = SPLITS_DIR / "train_v4_universal_balanced.csv"
val_csv   = SPLITS_DIR / "val_v4_universal_balanced.csv"
test_csv  = SPLITS_DIR / "test_v4_universal_balanced.csv"
test_bal_csv = SPLITS_DIR / "test_balanced.csv"

train_ds = UniversalDeepfakeDataset(train_csv, transform=train_transforms)
val_ds   = UniversalDeepfakeDataset(val_csv, transform=eval_transforms)
test_ds  = UniversalDeepfakeDataset(test_csv, transform=eval_transforms)
test_bal_ds = UniversalDeepfakeDataset(test_bal_csv, transform=eval_transforms)

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=CONFIG["num_workers"], pin_memory=(DEVICE.type == "cuda"), drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=CONFIG["batch_size"]*2, shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=(DEVICE.type == "cuda"))
test_loader  = DataLoader(test_ds, batch_size=CONFIG["batch_size"]*2, shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=(DEVICE.type == "cuda"))
test_bal_loader = DataLoader(test_bal_ds, batch_size=CONFIG["batch_size"]*2, shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=(DEVICE.type == "cuda"))

print(f"✅ Loaded Universal Balanced V4 Splits:")
print(f"  • Train V4 : {len(train_ds):,} samples ({len(train_loader)} batches)")
print(f"  • Val V4   : {len(val_ds):,} samples ({len(val_loader)} batches)")
print(f"  • Test V4  : {len(test_ds):,} samples ({len(test_loader)} batches)")
print(f"  • Test Bal : {len(test_bal_ds):,} samples ({len(test_bal_loader)} batches)")


✅ Loaded Universal Balanced V4 Splits:
  • Train V4 : 50,000 samples (3125 batches)
  • Val V4   : 5,000 samples (157 batches)
  • Test V4  : 5,000 samples (157 batches)
  • Test Bal : 4,134 samples (130 batches)


  ### 2. Lưu ý về tập thứ 4 (Test Bal - test_balanced.csv 4,134 mẫu):

  • test_balanced.csv là tập benchmark cũ (được cố định từ các giai đoạn đầu trước khi xây dựng bộ chia V4).
  • Vì train_v4 lấy mẫu từ toàn bộ kho dữ liệu lớn, nên giữa Train V4 và test_balanced.csv có trùng 1,166 mẫu.
  • Quy tắc đánh giá chuẩn:
      • Test V4 (test_v4_universal_balanced.csv) là tập Test độc lập chính thức (Zero-Leakage Ground Truth) để
      báo cáo kết quả đánh giá cuối cùng.
      • test_balanced.csv chỉ dùng để tham khảo so sánh cross-eval với các phiên bản thí nghiệm cũ (EXP-01, EXP-
      02).

In [6]:
# Step 3: Pretrained DINOv3 ViT Backbone & Enhanced Head Initialization
from src.models.dinov3_vit import load_dinov3

class EnhancedDinoViTClassifier(nn.Module):
    def __init__(self, backbone: nn.Module, num_classes: int = 2, hidden_dim: int = 384, dropout: float = 0.2):
        super().__init__()
        self.backbone = backbone
        embed_dim = backbone.embed_dim
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.backbone(x)
        return self.head(feat)

print("Loading Pretrained DINOv3 ViT Backbone...")
weights_path = PROJECT_ROOT / "models" / "dinov3-vits16plus-pretrain-lvd1689m" / "model-3.safetensors"
if not weights_path.exists():
    weights_path = PROJECT_ROOT / "experiments" / "checkpoints" / "weights" / "dinov3-vits16plus-pretrain-lvd1689m" / "model-3.safetensors"

print(f"Loading weights from: {weights_path}")
backbone = load_dinov3(str(weights_path), img_size=CONFIG["img_size"])

model = EnhancedDinoViTClassifier(backbone, num_classes=2, hidden_dim=CONFIG["hidden_dim"], dropout=CONFIG["dropout"]).to(DEVICE)
print(f"✅ Model built. Trainable Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


Loading Pretrained DINOv3 ViT Backbone...
Loading weights from: /workspace/hoangtuan/deepfake-ViT/models/dinov3-vits16plus-pretrain-lvd1689m/model-3.safetensors
✅ Model built. Trainable Parameters: 28,842,242


In [7]:
# Step 4: Layer-wise Learning Rate Decay (LLRD) & Cosine Scheduler
def build_llrd_param_groups(model: EnhancedDinoViTClassifier, base_lr: float, head_lr: float, decay_rate: float = 0.8, weight_decay: float = 0.05):
    param_groups = [{
        "params": [p for p in model.head.parameters() if p.requires_grad],
        "lr": head_lr,
        "weight_decay": weight_decay,
        "name": "head"
    }]
    num_layers = len(model.backbone.layer)
    for layer_idx in range(num_layers - 1, -1, -1):
        layer_lr = base_lr * (decay_rate ** (num_layers - 1 - layer_idx))
        block = model.backbone.layer[layer_idx]
        param_groups.append({
            "params": [p for p in block.parameters() if p.requires_grad],
            "lr": layer_lr,
            "weight_decay": weight_decay,
            "name": f"layer_{layer_idx}"
        })
    if hasattr(model.backbone, "norm"):
        param_groups.append({
            "params": [p for p in model.backbone.norm.parameters() if p.requires_grad],
            "lr": base_lr,
            "weight_decay": weight_decay,
            "name": "norm"
        })
    return param_groups

param_groups = build_llrd_param_groups(
    model, base_lr=CONFIG["base_lr"], head_lr=CONFIG["head_lr"],
    decay_rate=CONFIG["decay_rate"], weight_decay=CONFIG["weight_decay"]
)

optimizer = torch.optim.AdamW(param_groups)
total_steps = len(train_loader) * CONFIG["epochs"]
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-7)
criterion = LabelSmoothingCrossEntropy(smoothing=CONFIG["label_smoothing"]).to(DEVICE)
eval_criterion = nn.CrossEntropyLoss().to(DEVICE)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == "cuda"))
model_ema = ModelEMA(model, decay=CONFIG["ema_decay"])

print(f"✅ LLRD Optimizer initialized across {len(param_groups)} layer groups. Total Steps: {total_steps:,}")


✅ LLRD Optimizer initialized across 14 layer groups. Total Steps: 31,250


In [8]:
# Step 5: Multi-Metric Evaluation Engine
@torch.no_grad()
def evaluate_model(model, loader, device, criterion=None, use_tta=False):
    model.eval()
    all_y, all_prob, all_idx = [], [], []
    total_loss, n_batches = 0.0, 0

    for x, y, idxs in loader:
        x, y_dev = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        if use_tta:
            probs = predict_batch_with_tta(model, x, use_flips=True, use_multi_lighting=True)
        else:
            with torch.amp.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=(device.type == "cuda")):
                logits = model(x)
                if criterion:
                    loss = criterion(logits, y_dev)
                    total_loss += loss.item()
                    n_batches += 1
                probs = F.softmax(logits.float(), dim=-1)[:, 1]

        all_y.extend(y.tolist())
        all_prob.extend(probs.float().cpu().numpy().tolist())
        all_idx.extend(idxs.tolist())

    y_arr = np.array(all_y)
    prob_arr = np.array(all_prob)

    pred_05 = (prob_arr >= 0.5).astype(int)
    acc_05 = float(accuracy_score(y_arr, pred_05))
    try:
        auc = float(roc_auc_score(y_arr, prob_arr))
    except Exception:
        auc = 0.5

    try:
        fpr, tpr, thresholds = roc_curve(y_arr, prob_arr)
        j_scores = tpr - fpr
        opt_idx = np.argmax(j_scores)
        opt_tau = float(thresholds[opt_idx])
    except Exception:
        opt_tau = 0.5
        
    pred_opt = (prob_arr >= opt_tau).astype(int)
    acc_opt = float(accuracy_score(y_arr, pred_opt))
    val_loss = (total_loss / max(1, n_batches)) if (criterion and not use_tta) else 0.0

    return {
        'loss': val_loss,
        'accuracy': acc_05,
        'roc_auc': auc,
        'precision': float(precision_score(y_arr, pred_05, zero_division=0)),
        'recall': float(recall_score(y_arr, pred_05, zero_division=0)),
        'f1': float(f1_score(y_arr, pred_05, zero_division=0)),
        'opt_tau': opt_tau,
        'accuracy_opt': acc_opt,
        'f1_opt': float(f1_score(y_arr, pred_opt, zero_division=0)),
        'probs': prob_arr,
        'labels': y_arr,
        'indices': all_idx,
        'cm': confusion_matrix(y_arr, pred_05, labels=[0, 1]).tolist(),
    }

print("✅ Evaluation engine calibrated.")


✅ Evaluation engine calibrated.


In [ ]:
# Step 6: Interactive Training Execution Loop with EMA & Checkpointing
from tqdm.auto import tqdm

best_ckpt_path = CKPT_DIR / "best_checkpoint.pt"
latest_ckpt_path = CKPT_DIR / "latest_checkpoint.pt"

start_epoch = 1
best_val_auc = 0.0
history = []

print(f"🚀 Starting Interactive Training ({CONFIG['epochs']} Epochs)...")
for epoch in range(start_epoch, CONFIG["epochs"] + 1):
    model.train()
    train_loss, train_correct, total_train = 0.0, 0, 0
    t0 = time.time()

    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{CONFIG['epochs']:02d}", leave=True)
    for step, (images, labels, _) in enumerate(pbar):
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=DEVICE.type, dtype=torch.bfloat16, enabled=(DEVICE.type == "cuda")):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        scheduler.step()
        model_ema.update(model)

        train_loss += loss.item() * len(labels)
        preds = logits.argmax(dim=-1)
        train_correct += (preds == labels).sum().item()
        total_train += len(labels)

        current_loss = train_loss / total_train
        current_acc = train_correct / total_train * 100.0
        pbar.set_postfix({
            "loss": f"{current_loss:.4f}",
            "acc": f"{current_acc:.2f}%",
            "lr": f"{optimizer.param_groups[0]['lr']:.2e}"
        })

    epoch_time = time.time() - t0
    train_loss = train_loss / total_train
    train_acc = train_correct / total_train * 100.0

    # Validation evaluation using EMA model
    val_metrics = evaluate_model(model_ema.module, val_loader, DEVICE, criterion=eval_criterion)
    val_acc = val_metrics["accuracy"] * 100.0
    val_auc = val_metrics["roc_auc"]
    val_f1 = val_metrics["f1"]
    opt_acc = val_metrics["accuracy_opt"] * 100.0

    print(f"📊 Epoch {epoch:02d} ({epoch_time:.1f}s) | Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_metrics['loss']:.4f} Acc: {val_acc:.2f}% (Opt: {opt_acc:.2f}%) | "
          f"Val AUC: {val_auc:.4f} | F1: {val_f1:.4f}")

    epoch_record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_metrics["loss"],
        "val_acc": val_acc,
        "val_auc": val_auc,
        "val_f1": val_f1,
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "opt_tau": val_metrics["opt_tau"],
        "opt_acc": opt_acc,
        "epoch_time": epoch_time
    }
    history.append(epoch_record)

    state = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "ema_state": model_ema.module.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state": scaler.state_dict(),
        "best_val_auc": best_val_auc,
        "config": CONFIG,
        "history": history
    }
    torch.save(state, latest_ckpt_path)

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        state["best_val_auc"] = best_val_auc
        torch.save(state, best_ckpt_path)
        print(f"  ⭐ New Best Checkpoint saved! (Val AUC: {best_val_auc:.4f})")


🚀 Starting Interactive Training (10 Epochs)...


Epoch 01/10:   0%|          | 0/3125 [00:00<?, ?it/s]

📊 Epoch 01 (855.1s) | Train Loss: 0.4035 Acc: 83.68% | Val Loss: 0.2826 Acc: 88.68% (Opt: 88.74%) | Val AUC: 0.9547 | F1: 0.8843
  ⭐ New Best Checkpoint saved! (Val AUC: 0.9547)


Epoch 02/10:   0%|          | 0/3125 [00:00<?, ?it/s]

In [ ]:
# Step 7: Training History & Convergence Curves
if len(history) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    epochs_range = [h["epoch"] for h in history]

    # Loss
    axes[0].plot(epochs_range, [h["train_loss"] for h in history], 'o-', label="Train Loss", color="royalblue", lw=2)
    axes[0].plot(epochs_range, [h["val_loss"] for h in history], 's--', label="Val Loss", color="crimson", lw=2)
    axes[0].set_title("Loss Convergence", fontsize=13, fontweight="bold")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    # Accuracy
    axes[1].plot(epochs_range, [h["train_acc"] for h in history], 'o-', label="Train Acc", color="royalblue", lw=2)
    axes[1].plot(epochs_range, [h["val_acc"] for h in history], 's--', label="Val Acc (0.50)", color="crimson", lw=2)
    axes[1].plot(epochs_range, [h["opt_acc"] for h in history], '^-.', label="Val Acc (Opt Tau)", color="green", lw=2)
    axes[1].set_title("Accuracy Convergence (%)", fontsize=13, fontweight="bold")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy (%)")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    # ROC-AUC
    axes[2].plot(epochs_range, [h["val_auc"] for h in history], 'd-', label="Val ROC-AUC", color="darkorange", lw=2)
    axes[2].set_title("Validation ROC-AUC", fontsize=13, fontweight="bold")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("AUC")
    axes[2].grid(True, alpha=0.3)
    axes[2].legend()

    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "exp04_v4universal_convergence_curves.png", dpi=300)
    plt.show()


In [ ]:
# Step 8: Validation V4 Multi-Metric Evaluation & Confusion Matrix
print("Evaluating Best Model on Validation V4 Set (5,000 samples)...")
if best_ckpt_path.exists():
    best_ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
    model.load_state_dict(best_ckpt["ema_state"])

val_results = evaluate_model(model, val_loader, DEVICE, criterion=eval_criterion)

print(f"📊 VALIDATION V4 RESULTS:")
print(f"  • Val Loss       : {val_results['loss']:.4f}")
print(f"  • Val Accuracy   : {val_results['accuracy']*100:.2f}% (Threshold 0.50)")
print(f"  • Optimal Tau    : {val_results['opt_tau']:.4f} -> Optimal Accuracy: {val_results['accuracy_opt']*100:.2f}%")
print(f"  • Val ROC-AUC    : {val_results['roc_auc']:.4f}")
print(f"  • Val F1-Score   : {val_results['f1']:.4f}")
print(f"  • Confusion Mat  : Real={val_results['cm'][0]}, Fake={val_results['cm'][1]}")


In [ ]:
# Step 9: Independent Test V4 Benchmark Evaluation & TTA
print("Evaluating on Independent Test V4 Set (5,000 samples)...")
test_v4_std = evaluate_model(model, test_loader, DEVICE, criterion=eval_criterion, use_tta=False)
test_v4_tta = evaluate_model(model, test_loader, DEVICE, criterion=eval_criterion, use_tta=True)

print(f"📊 TEST V4 (ZERO-LEAKAGE INDEPENDENT TEST SET):")
print(f"  • Standard (No TTA) : Acc: {test_v4_std['accuracy']*100:.2f}% | ROC-AUC: {test_v4_std['roc_auc']:.4f} | F1: {test_v4_std['f1']:.4f}")
print(f"  • Enhanced (+ TTA)  : Acc: {test_v4_tta['accuracy']*100:.2f}% | ROC-AUC: {test_v4_tta['roc_auc']:.4f} | F1: {test_v4_tta['f1']:.4f}")


In [ ]:
# Step 10: Frozen International Test Balanced Benchmark (4,134 samples)
print("Evaluating on Frozen Test Balanced Benchmark (4,134 samples)...")
test_bal_std = evaluate_model(model, test_bal_loader, DEVICE, criterion=eval_criterion, use_tta=False)
test_bal_tta = evaluate_model(model, test_bal_loader, DEVICE, criterion=eval_criterion, use_tta=True)

print(f"📊 FROZEN TEST BALANCED BENCHMARK RESULTS:")
print(f"  • Standard (No TTA) : Acc: {test_bal_std['accuracy']*100:.2f}% | ROC-AUC: {test_bal_std['roc_auc']:.4f} | F1: {test_bal_std['f1']:.4f}")
print(f"  • Enhanced (+ TTA)  : Acc: {test_bal_tta['accuracy']*100:.2f}% | ROC-AUC: {test_bal_tta['roc_auc']:.4f} | F1: {test_bal_tta['f1']:.4f}")


In [ ]:
# Step 11: Granular 38-Method Breakdown Table & Hard-Method Plot
test_methods = test_ds.methods
per_method_records = []

for m in sorted(list(set(test_methods))):
    mask = (test_methods == m)
    if mask.sum() > 0:
        m_labels = test_v4_tta["labels"][mask]
        m_probs = test_v4_tta["probs"][mask]
        m_preds = (m_probs >= test_v4_tta["opt_tau"]).astype(int)
        m_acc = accuracy_score(m_labels, m_preds) * 100.0
        per_method_records.append({
            "Method / Domain": m,
            "Type": "Real" if "Real" in m else "Fake",
            "Sample Count": int(mask.sum()),
            "Accuracy (TTA)": f"{m_acc:.2f}%",
            "Mean Prob Fake": f"{np.mean(m_probs):.4f}"
        })

df_method_perf = pd.DataFrame(per_method_records).sort_values(by=["Type", "Method / Domain"]).reset_index(drop=True)
method_perf_csv = RESULTS_DIR / "exp04_test_v4_method_breakdown.csv"
df_method_perf.to_csv(method_perf_csv, index=False)

print(f"📊 GRANULAR METHOD BREAKDOWN SAVED TO: {method_perf_csv}\n")
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)
print(df_method_perf.to_string(index=False))
